In [1]:
# %%
import os
import sys
import json
import re # <-- Added
from pathlib import Path

# --- Make sure your utils are imported for pathing ---
file_path = os.getcwd()
project_root = os.path.dirname(os.path.dirname(file_path))
print(f"Project Root: {project_root}")
if project_root not in sys.path:
    sys.path.append(project_root)

from dotenv import load_dotenv
import chromadb
from langchain_core.embeddings import Embeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_ollama import OllamaEmbeddings
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma 
from langchain_core.documents import Document

# --- Import your project's agents and utils ---
from crunch_llm.code.agents import BasicAgent 
from crunch_llm.code.utils import initialize_local_data


Project Root: /oscar/data/gk/jdtoscan/LLM/Crunch_LLM


/users/jdtoscan/.conda/envs/Agent_Hub/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
analyzer_system_prompt_scic = """
You are an expert Scientific Machine Learning (SciML) code analyst. Your task is to analyze the provided code snippet (typically PyTorch, JAX, or TensorFlow) and extract key information into a structured JSON format. You must be highly specific and infer the values from the model definitions, loss functions, and training loops.

Your final output must be a single, valid JSON object. Do not add any explanatory text outside the JSON.

### Fields to Extract

1.  **`method`**: Identify the specific neural architecture or operator learning scheme.
    * *Examples*: `Physics-Informed Neural Network (PINN)`, `Kolmogorov-Arnold Network (KAN)`, `Deep Operator Network (DeepONet)`, `Fourier Neural Operator (FNO)`, `Weak-Form PINN (VarPINN)`.
    * *Hints*: Look for class names like `KANLayer`, `DeepONet`, `SpectralConv2d`, or standard MLPs used in a physics loop.

2.  **`equation`**: Identify the governing PDE being solved.
    * *Examples*: `Inviscid Burgers`, `Navier-Stokes`, `Allen-Cahn`, `Wave Equation`.
    * *Hints*: Look for the residual function (e.g., `def pde(u, x): return u_t + u*u_x`), or variable names like `nu` (viscosity), `rho`.

3.  **`dimensionality`**: Specify spatial/temporal dimensions and the domain type.
    * *Examples*: `1D Spatio-Temporal`, `2D Steady-State`, `3D Forward`.
    * *Hints*: Check input tensor shapes (e.g., `inputs.shape[1] == 2` implies 1D+t or 2D), or the coordinate limits `x \in [-1,1]`.

4.  **`solver_or_optimizer`**: Identify the optimization strategy.
    * *Examples*: `Adam`, `L-BFGS`, `Hybrid (Adam + SSBroyden)`, `Hybrid (Adam + L-BFGS)`.
    * *Hints*: Look for `torch.optim.Adam`, `LBFGS`, or custom closure calls involving `SSBroyden` or `Newton`. Note if a switch happens (e.g., "Train with Adam then switch to LBFGS").

5.  **`enhancements`**: List specific SciML tricks found in the code.
    * *Examples*: `["Periodic Fourier Features"]`, `["Hard Constraint Ansatz"]`, `["Adaptive Activation (Rowdy)"]`, `["Self-Adaptive Loss Weights (GradNorm)"]`, `["Causal Training"]`.
    * *Hints*: Look for `sin/cos` embeddings, distance functions (`x * (1-x)`), trainable parameters in the loss (e.g., `log_var`), or masks applied to time coordinates.

6.  **`BCs`**: Identify how Boundary/Initial Conditions are enforced.
    * *Examples*: `Soft Constraints (Loss-based)`, `Hard Constraints (Ansatz)`, `Periodic (Fourier Embedding)`.
    * *Hints*: Look for `mse_bc` in the total loss (Soft), or a forward pass like `u = g(x) + t*N(x)` (Hard), or `sin(k*pi*x)` features (Periodic).

7.  **`problem`**: Describe the physical task (Forward vs Inverse).
    * *Examples*: `Forward solution of Burgers equation`, `Inverse identification of viscosity parameter`.
    * *Hints*: If `epsilon` or `nu` is a `nn.Parameter` or `requires_grad=True`, it is an **Inverse** problem. Otherwise, it is **Forward**.

8.  **`description`**: Provide a concise 1-2 sentence summary of the code's methodology.

---

### Example 1: Standard PINN with Hybrid Optimization

*Input Code Snippet:*
```python
class PINN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2, 50), nn.Tanh(), nn.Linear(50, 1))
    def forward(self, x, t):
        # Hard constraint for IC: u(x,0) = -sin(pi*x)
        return -torch.sin(np.pi*x) + t * self.net(torch.cat([x, t], dim=1))

# Optimizer Logic
optimizer1 = torch.optim.Adam(model.parameters(), lr=1e-3)
# ... training loop 1 ...
optimizer2 = SSBroyden(model.parameters(), lr=1) # Quasi-Newton refinement
# ... training loop 2 ...
```

*Your Required JSON Output:*
```json
{{
  "method": "Physics-Informed Neural Network (PINN)",
  "equation": "General 1D PDE (Likely Burgers/Transport based on sine IC)",
  "dimensionality": "1D Spatio-Temporal",
  "solver_or_optimizer": "Hybrid (Adam + SSBroyden)",
  "enhancements": [
    "Hard Constraints (Ansatz)"
  ],
  "BCs": "Hard Constraints (Ansatz)",
  "problem": "Forward solution of a 1D PDE.",
  "description": "A PINN solving a 1D time-dependent PDE using a hard-constraint ansatz to satisfy the Initial Condition by design, trained via a Hybrid Adam + SSBroyden strategy."
}}
```

### Example 2: KAN with Fourier Features (Inverse Problem)

*Input Code Snippet:*
```python
class PhysicsKAN(nn.Module):
    def __init__(self):
        self.embed = FourierFeature(input_dim=2, scale=1.0) # Random Fourier Features
        self.kan = KANLayer(in_dim=10, out_dim=1)
        self.viscosity = nn.Parameter(torch.tensor(0.0)) # Unknown parameter

    def loss(self, x):
        u = self.kan(self.embed(x))
        u_t, u_x, u_xx = grad(u, x)
        # Inverse problem logic
        return torch.mean((u_t + u*u_x - self.viscosity * u_xx)**2) 
```

*Your Required JSON Output:*
```json
{{
  "method": "Kolmogorov-Arnold Network (KAN)",
  "equation": "Viscous Burgers Equation",
  "dimensionality": "1D Spatio-Temporal",
  "solver_or_optimizer": "Unknown (Likely Adam based on context)",
  "enhancements": [
    "Random Fourier Features",
    "KAN Architecture"
  ],
  "BCs": "Soft Constraints (Loss-based)",
  "problem": "Inverse identification of viscosity parameter.",
  "description": "A Kolmogorov-Arnold Network (KAN) enhanced with Fourier Features, designed to identify the unknown viscosity parameter in the Burgers equation."
}}
```
"""

In [3]:

# 1. Define the exact path where you want to save the database
my_database_path ="/users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Documents/database_PIML"

COLLECTION_NAME = 'PINNs_Codes'

# 2. Pass that path to your function
vector_store = initialize_local_data(db_path=my_database_path,collection_name=COLLECTION_NAME)

analyzer_agent = BasicAgent(
                     model='gemini-2.5-flash', 
                     model_provider='google',
                     system_prompt=analyzer_system_prompt_scic,
                     verbose=False)


Initializing with Ollama. Model: bge-m3:567m
✅ Vector store initialized successfully from path: /users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Documents/database_PIML
   Provider: ollama, Model: bge-m3:567m, Collection: PINNs_Codes


In [6]:
import os
import re
import json
from pathlib import Path
from langchain_core.documents import Document

# The directory containing your code files
target_path = '/users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Documents/codes/SciML/PINNs'

all_results = {}
print(f"\nStarting analysis of all files in: {target_path}")

# os.walk will traverse the entire directory tree
for root, dirs, files in os.walk(target_path):
    for filename in files:
        file_path = os.path.join(root, filename)
        
        # Skip non-python/julia files (Added .py just in case)
        if not filename.endswith('.jl') and not filename.endswith('.py'):
            continue
            
        try:
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                code_content = f.read()
            line_count = code_content.count('\n') + 1
            print(f"Processing: {filename} ({line_count} lines)...")
            
            # --- 1. Run Analyzer Agent ONCE on the full file ---
            print("  - Running Analyzer Agent...")
            
            # Call Agent (Assuming it returns a dict with 'output')
            agent_result = analyzer_agent.run(code_content)
            
            # Handle response types (Dict vs String)
            if isinstance(agent_result, dict) and 'output' in agent_result:
                summary_string = agent_result['output'].strip()
            else:
                summary_string = str(agent_result).strip()
            
            # --- START FIX: Parse JSON and Flatten List ---
            try:
                # Clean markdown fences if the agent added them
                clean_json = summary_string.replace("```json", "").replace("```", "").strip()
                data = json.loads(clean_json)

                analysis_result = {
                    # Cast everything to string and flatten lists
                    "method": str(data.get("method", "Unknown")),
                    "equation": str(data.get("equation", "Unknown")),
                    "dimensionality": str(data.get("dimensionality", "Unknown")),
                    "solver_or_optimizer": str(data.get("solver_or_optimizer", "Unknown")),
                    
                    # CRITICAL FIX: Join the list into a single string
                    "enhancements": ", ".join(data.get("enhancements", [])),
                    
                    "BCs": str(data.get("BCs", "Unknown")),
                    "problem": str(data.get("problem", "Unknown")),
                    "description": str(data.get("description", "Unknown")),
                    "filename": filename
                }
            except Exception as e:
                print(f"  - Warning: Could not parse JSON '{summary_string[:50]}...'. Error: {e}")
                analysis_result = {
                    "method": "ParseError",
                    "equation": "Unknown",
                    "enhancements": "",
                    "filename": filename
                }
            # --- END FIX ---

            print(f"  - Analysis: {analysis_result['method']} | {analysis_result['equation']}")
            
            
            # --- 2. Correct Splitting & Re-assembly ---
            print("  - Splitting code into cells...")
            
            # This pattern matches '# ## Cell N:' at the START of any line
            split_pattern = r'(^\s*[#\s]+Cell\s*\d+:.*)'
            
            # Split using MULTILINE mode
            parts = re.split(split_pattern, code_content, flags=re.MULTILINE)
            
            cleaned_cells = []
            
            # Handle the "prologue" (anything before the first Cell marker)
            if parts[0].strip():
                cleaned_cells.append(parts[0].strip())
                
            # Loop over the split parts, taking 2 at a time
            for i in range(1, len(parts), 2):
                if i + 1 < len(parts):
                    full_cell = parts[i] + parts[i+1]
                    cleaned_cells.append(full_cell.strip())
                else:
                    cleaned_cells.append(parts[i].strip())

            num_cells = len(cleaned_cells)
            
            print(f"  - Found {num_cells} cells. Ingesting each as a 'part'...")

            # --- 3. Prepare common metadata & header info ---
            base_metadata = {
                'source_file': file_path,
                'filename': filename,
                'type': 'code-notebook',
                **analysis_result # Add the flattened analysis
            }
            
            file_ext = Path(file_path).suffix.lower()
            language_map = {'.py': 'python', '.m': 'matlab', '.jl': 'julia', '.cpp': 'cpp', '.c': 'c'}
            language = language_map.get(file_ext, 'unknown')
            comment_styles = {'python': '#', 'matlab': '%', 'julia': '#', 'c': '//', 'cpp': '//'}
            comment = comment_styles.get(language, '#')
            
            # Create a pretty string for the header (for reading)
            analysis_str = json.dumps(analysis_result, indent=2).replace('{', '').replace('}', '').replace('"', '')

            docs_to_add = []
            
            # --- 4. Loop through each cell and create a Document ---
            for i, cell_content in enumerate(cleaned_cells):
                part_num = i + 1
                
                # --- Extract the cell's description ---
                cell_description = "File Prologue / Imports" 
                
                if cell_content.startswith('# ## Cell'):
                     first_line = cell_content.split('\n', 1)[0]
                     cell_description = first_line.strip().lstrip('#').strip()
                else:
                    first_line = ""
                    for line in cell_content.split('\n'):
                        if line.strip() and not line.strip().startswith('#'):
                            first_line = line
                            break
                    if first_line:
                        cell_description = first_line.strip()
                
                # --- Create metadata for this chunk ---
                chunk_metadata = base_metadata.copy()
                chunk_metadata['part'] = f"{part_num}/{num_cells}"
                chunk_metadata['cell_description'] = cell_description
                
                # --- Create the full header for this chunk ---
                header = f"""{comment} CODE ANALYSIS (PART {part_num} of {num_cells})
{comment} DESCRIPTION:
{comment} 
{comment} {analysis_str.replace(chr(10), f'{chr(10)}{comment} ')}
{comment} =======================================================\n\n"""
                
                enhanced_content = header + cell_content
                
                doc = Document(page_content=enhanced_content, metadata=chunk_metadata)
                docs_to_add.append(doc)

            # --- 5. Add all documents for this file at once ---
            if docs_to_add:
                vector_store.add_documents(docs_to_add)
                
            print(f"✅ Ingested: {filename} ({num_cells} chunks)")
            all_results[file_path] = {"analysis": analysis_result, "chunks_created": num_cells}

        except Exception as e:
            print(f"--- UNHANDLED ERROR analyzing {filename}: {e} ---")
            all_results[file_path] = {"error": str(e)}

# --- 6. Save and Display Results ---
output_file = 'full_code_analysis.json'
print(f"\nAnalysis complete! ✅ Saving all results to {output_file}...")
with open(output_file, 'w') as f:
    json.dump(all_results, f, indent=4)
print("Done.")


Starting analysis of all files in: /users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Documents/codes/SciML/PINNs
Processing: MLP_SSBroyden.py (784 lines)...
  - Running Analyzer Agent...
  - Analysis: Physics-Informed Neural Network (PINN) | Allen-Cahn Equation
  - Splitting code into cells...
  - Found 19 cells. Ingesting each as a 'part'...
✅ Ingested: MLP_SSBroyden.py (19 chunks)

Analysis complete! ✅ Saving all results to full_code_analysis.json...
Done.


In [5]:

# Assuming 'vector_store' is your initialized Chroma object
print("Fetching all documents from the vector store...")

# The .get() method returns a dictionary with IDs, documents, metadatas, etc.
# We specify what information we want to retrieve.
all_entries = vector_store.get(
    include=["metadatas", "documents"] # You can also include "embeddings"
)

# Let's print the first 5 documents found
num_entries = len(all_entries['ids'])
print(f"Found {num_entries} total entries.\n")

for i in range(min(5, num_entries)):
    doc_id = all_entries['ids'][i]
    metadata = all_entries['metadatas'][i]
    content = all_entries['documents'][i]
    
    print(f"--- Entry {i+1} ---")
    print(f"ID: {doc_id}")
    print(f"Source File: {metadata.get('source_file', 'N/A')}")
    print(f"Content Preview: {content[:15000]}...") # Print first 150 chars
    print("-" * 20)


Fetching all documents from the vector store...
Found 0 total entries.



In [4]:
import chromadb
import os
from pathlib import Path
import sys

# --- Make sure your utils are imported for pathing ---
# (This isn't strictly necessary but good for consistency)
file_path = os.getcwd()
project_root = os.path.dirname(file_path)
if project_root not in sys.path:
    sys.path.append(project_root)

my_collection_name = COLLECTION_NAME

print(f"Attempting to clear elements from collection: '{my_collection_name}'")
print(f"From database path: {my_database_path}")

try:
    # --- 2. INITIALIZE THE CLIENT ---
    client = chromadb.PersistentClient(path=my_database_path)
    
    # --- 3. GET THE COLLECTION ---
    collection = client.get_collection(name=my_collection_name)
    
    # --- 4. GET ALL ELEMENT IDS ---
    # We get all items. include=[] is efficient as we only need IDs.
    all_items = collection.get(include=[]) 
    all_ids = all_items['ids']
    
    if not all_ids:
        print(f"\nInfo: Collection '{my_collection_name}' is already empty.")
    else:
        # --- 5. DELETE ALL ELEMENTS BY ID ---
        print(f"\nFound {len(all_ids)} elements. Deleting all elements...")
        collection.delete(ids=all_ids)
        print(f"✅ Success: All {len(all_ids)} elements have been deleted from '{my_collection_name}'.")
        print("The collection itself still exists.")

except chromadb.errors.CollectionNotFoundError:
    print(f"\nInfo: Collection '{my_collection_name}' does not exist. Nothing to delete.")
except Exception as e:
    print(f"\nAn error occurred: {e}")
    print("Please check the path and ChromaDB server status.")



Attempting to clear elements from collection: 'PINNs_Codes'
From database path: /users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Documents/database_PIML

Found 19 elements. Deleting all elements...
✅ Success: All 19 elements have been deleted from 'PINNs_Codes'.
The collection itself still exists.
